In [2]:
from torchvision.models import ResNet18_Weights

weights = ResNet18_Weights.DEFAULT

print(weights.transforms())

ImageClassification(
    crop_size=[224]
    resize_size=[256]
    mean=[0.485, 0.456, 0.406]
    std=[0.229, 0.224, 0.225]
    interpolation=InterpolationMode.BILINEAR
)


In [3]:
from PIL import Image
from pathlib import Path

preprocess = weights.transforms()

data_dir = Path("../data/bottle")
train_images = list((data_dir / "train" / "good").glob("*.png"))
image_path = train_images[0]
image = Image.open(image_path)

image_tensor = preprocess(image)
print(image_tensor.shape)
print(image_tensor.dtype)
print(image_tensor.min())
print(image_tensor.max())


torch.Size([3, 224, 224])
torch.float32
tensor(-1.5870)
tensor(2.6400)


In [4]:
batch = image_tensor.unsqueeze(0)

print(batch.shape)

torch.Size([1, 3, 224, 224])


In [5]:
from torchvision.models import resnet18

model = resnet18(weights=weights)
print(model)

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_sta

In [6]:
import torch.nn as nn
import torch

model.fc = nn.Identity()

model.eval()

with torch.no_grad():
    features = model(batch)

print(features.shape)


torch.Size([1, 512])


In [7]:
from visual_inspection.features import create_feature_extractor, extract_image_features
from pathlib import Path

model, preprocess = create_feature_extractor()

features = extract_image_features(
    Path("../data/bottle/train/good/000.png"),
    model,
    preprocess,
)

print(features.shape)

torch.Size([1, 512])


In [8]:
from visual_inspection.data import (
    build_dataset_index,
    split_normal_train_validation,
    validate_dataset_index,
)

data_dir = Path("../data/bottle")

record_df = build_dataset_index(data_dir)
validate_dataset_index(record_df)

train_df, validation_df = split_normal_train_validation(
    record_df,
    validation_size=0.2,
    random_state=42,
)

print(f"Train samples: {len(train_df)}")
print(f"Validation samples: {len(validation_df)}")

Train samples: 167
Validation samples: 42


In [9]:
from visual_inspection.features import extract_dataset_features

train_features = extract_dataset_features(
    train_df,
    model,
    preprocess,
)

validation_features = extract_dataset_features(
    validation_df,
    model,
    preprocess,
)

print(train_features.shape)
print(validation_features.shape)

torch.Size([167, 512])
torch.Size([42, 512])
